# Agrupamento de Notícias com Análise Semântica via LLM

**Disciplina:** Processamento de Linguagem Natural / Aprendizado de Máquina  
**Orientador:** Prof. José Alfredo F. Costa (DCA/UFRN)  
**Autores:** Cauã Vitor, Ivan  
**Dataset:** `Base_dados_textos_6_classes.csv` — 315 notícias, 6 categorias (PT-BR)

---

## Objetivo

Executar e documentar de forma **reprodutível** experimentos de clustering não-supervisionado sobre um corpus de notícias em português,
comparando representações textuais clássicas (TF-IDF) com embeddings semânticos densos (Sentence Transformers PT-BR).

### Estrutura do Notebook

| Seção | Conteúdo |
|:---:|:---|
| 1 | Configuração e importação de bibliotecas |
| 2 | Carga, limpeza e análise exploratória dos dados |
| 3 | Extração de atributos: TF-IDF vs Embeddings densos |
| 4 | Clustering com 4 algoritmos e cálculo de métricas |
| 5 | Visualização 2D com PCA, t-SNE e UMAP |
| 6 | Seleção de landmarks (medoides e fronteiras) |

---

## 1. Configuração e Importação de Bibliotecas

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from dotenv import load_dotenv

# Scikit-Learn — ML e métricas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score, silhouette_samples, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.preprocessing import LabelEncoder

# Embeddings locais
from sentence_transformers import SentenceTransformer
import umap

warnings.filterwarnings('ignore')
load_dotenv('backend/.env')

# Paleta de cores fixas para as 6 categorias
CATEGORY_COLORS = {
    'Economia':              '#7aa2f7',  # blue
    'Esportes':              '#ff9e64',  # orange
    'Polícia e Direitos':    '#f7768e',  # red
    'Política':              '#bb9af7',  # purple
    'Turismo':               '#7dcfff',  # cyan
    'Variedades e Sociedade':'#e0af68',  # yellow
}
CLUSTER_CMAP = plt.get_cmap('tab10')

print('✅ Bibliotecas importadas com sucesso!')

## 2. Carga, Limpeza e Análise Exploratória

O arquivo CSV é separado por ponto-e-vírgula e contém 4 colunas principais:
- `Texto Original`: título/headline da notícia
- `Texto Expandido`: corpo completo
- `Classe`: rótulo numérico da categoria
- `Categoria`: nome da categoria (ground truth)

Removemos registros com valores nulos para garantir que os algoritmos de vetorização funcionem corretamente.

In [ ]:
csv_path = 'Base_dados_textos_6_classes.csv'
df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
print(f'Tamanho original: {len(df)} registros')

# Limpeza de nulos
df = df.dropna(subset=['Texto Original', 'Texto Expandido']).reset_index(drop=True)
print(f'Após limpeza:    {len(df)} registros\n')

# Distribuição das categorias
print('Distribuição por Categoria Real:')
print(df['Categoria'].value_counts().to_frame('count').to_string())

# Codificar labels para uso com sklearn
true_labels = df['Classe'].values

df.head(3)

In [ ]:
# Visualização da distribuição das classes
fig, ax = plt.subplots(figsize=(9, 4), dpi=120)
counts = df['Categoria'].value_counts()
bars = ax.barh(counts.index, counts.values,
               color=[CATEGORY_COLORS.get(c, '#888') for c in counts.index],
               edgecolor='white', linewidth=0.5)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_xlabel('Número de amostras', fontsize=11)
ax.set_title('Distribuição das 6 Categorias — Base de Notícias PT-BR', fontsize=13, fontweight='bold')
ax.set_xlim(0, counts.max() * 1.18)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Extração de Atributos — TF-IDF vs Embeddings Semânticos Densos

Comparamos duas estratégias de representação:

### 3.1 TF-IDF (Term Frequency – Inverse Document Frequency)

Abordagem clássica *bag-of-words*. Cada documento vira um vetor esparso de pesos:

$$\text{TF-IDF}(t, d, D) = \text{tf}(t, d) \times \log\!\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

**Limitação:** ignora contexto semântico — "banco" (financeiro) e "banco" (assento) teriam o mesmo vetor.

### 3.2 Sentence Transformers (Embeddings Densos PT-BR)

Modelo `paraphrase-multilingual-MiniLM-L12-v2` — treinado em múltiplos idiomas incluindo PT-BR. Gera vetores densos de 384 dimensões que capturam semântica, sinônimos e relações implícitas via atenção multi-cabeça (transformers).

In [ ]:
texts = df['Texto Expandido'].tolist()

# Lista extensa de stopwords PT-BR
PT_STOPWORDS = [
    'a','ao','aos','as','ate','com','como','da','das','de','dela','delas','dele',
    'deles','do','dos','e','ela','elas','ele','eles','em','entre','essa','esse',
    'esta','este','eu','foi','fosse','ha','ja','mas','me','meu','minha','muito',
    'na','nas','no','nos','o','os','ou','para','pela','pelo','por','que','quem',
    'se','sem','ser','seu','seus','so','sua','suas','tambem','te','tem','temos',
    'um','uma','uns','voce','voces'
]

# --- TF-IDF ---
print('⏳ Gerando TF-IDF...')
t0 = time.time()
vectorizer = TfidfVectorizer(max_features=1000, stop_words=PT_STOPWORDS)
X_tfidf = vectorizer.fit_transform(texts).toarray()
print(f'   Matriz TF-IDF: {X_tfidf.shape}  ({time.time()-t0:.2f}s)\n')

# --- Sentence Transformers PT-BR ---
print('⏳ Gerando Embeddings Densos (paraphrase-multilingual-MiniLM-L12-v2)...')
t0 = time.time()
st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
X_dense = st_model.encode(texts, show_progress_bar=True, batch_size=32)
print(f'   Embeddings densos: {X_dense.shape}  ({time.time()-t0:.2f}s)')

## 4. Clustering e Métricas de Avaliação

### Métricas Avaliadas

| Métrica | Tipo | Interpretação |
|:---|:---:|:---|
| **Silhueta** $s(i) \in [-1, +1]$ | Não supervisionada | Coesão intra-cluster vs. separação inter-cluster. Maior = melhor. |
| **Davies-Bouldin** | Não supervisionada | Razão compacidade/separação. Menor = melhor. |
| **ARI** $\in [-1, 1]$ | Supervisionada | Concordância com ground truth, corrigida pelo acaso. |
| **NMI** $\in [0, 1]$ | Supervisionada | Informação mútua normalizada com as classes reais. |
| **Pureza** $\in [0, 1]$ | Supervisionada | Fração das amostras na classe majoritária de cada cluster. |

A **Silhueta** do ponto $i$ é:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
onde $a(i)$ é a distância média intra-cluster e $b(i)$ é a distância média ao cluster vizinho mais próximo.

In [ ]:
def purity_score(true_labels, pred_labels):
    """Calcula a pureza do clustering ignorando pontos de ruído (-1)."""
    total = len(true_labels)
    match_sum = 0
    for cluster in set(pred_labels):
        mask = pred_labels == cluster
        if cluster == -1:
            match_sum += mask.sum()  # Cada ruído é sua própria classe
            continue
        cluster_true = true_labels[mask]
        match_sum += pd.Series(cluster_true).value_counts().iloc[0]
    return match_sum / total


def evaluate_clustering(X, true_labels, pred_labels, name=''):
    """Calcula todas as métricas de avaliação para um conjunto de labels."""
    pred_labels = np.array(pred_labels)
    n_clusters = len(set(pred_labels) - {-1})
    noise = (pred_labels == -1).sum()

    if n_clusters > 1 and n_clusters < len(pred_labels):
        sil = silhouette_score(X, pred_labels)
        db  = davies_bouldin_score(X, pred_labels)
    else:
        sil, db = -1.0, -1.0

    ari    = adjusted_rand_score(true_labels, pred_labels)
    nmi    = normalized_mutual_info_score(true_labels, pred_labels)
    purity = purity_score(np.array(true_labels), pred_labels)

    return {
        'Config': name,
        'K':      n_clusters,
        'Ruído':  int(noise),
        'Silhueta':       round(sil, 4),
        'Davies-Bouldin': round(db, 4),
        'ARI':    round(ari, 4),
        'NMI':    round(nmi, 4),
        'Pureza': round(purity, 4),
    }


# -----------------------------------------------------------------------
# Experimentos: K-Means, Agglomerative, DBSCAN e HDBSCAN em TF-IDF e ST
# -----------------------------------------------------------------------
results = []

# -- TF-IDF --
km_tfidf   = KMeans(n_clusters=6, random_state=42, n_init=10)
agg_tfidf  = AgglomerativeClustering(n_clusters=6, linkage='ward')

# Normaliza para DBSCAN (cosine ≈ euclidean em vetores normalizados)
X_tfidf_norm = X_tfidf / (np.linalg.norm(X_tfidf, axis=1, keepdims=True) + 1e-12)
db_tfidf   = DBSCAN(eps=1.15, min_samples=3)
hdb_tfidf  = HDBSCAN(min_cluster_size=5, min_samples=3)

results.append(evaluate_clustering(X_tfidf, true_labels, km_tfidf.fit_predict(X_tfidf),     'TF-IDF + K-Means'))
results.append(evaluate_clustering(X_tfidf, true_labels, agg_tfidf.fit_predict(X_tfidf),    'TF-IDF + Agglomerative'))
results.append(evaluate_clustering(X_tfidf_norm, true_labels, db_tfidf.fit_predict(X_tfidf_norm),   'TF-IDF + DBSCAN'))
results.append(evaluate_clustering(X_tfidf, true_labels, hdb_tfidf.fit_predict(X_tfidf),    'TF-IDF + HDBSCAN'))

# -- Sentence Transformers (PT-BR) --
km_st   = KMeans(n_clusters=6, random_state=42, n_init=10)
agg_st  = AgglomerativeClustering(n_clusters=6, linkage='ward')
X_st_norm = X_dense / (np.linalg.norm(X_dense, axis=1, keepdims=True) + 1e-12)
db_st   = DBSCAN(eps=0.4, min_samples=3)
hdb_st  = HDBSCAN(min_cluster_size=5, min_samples=3)

results.append(evaluate_clustering(X_dense, true_labels, km_st.fit_predict(X_dense),       'ST (PT-BR) + K-Means'))
results.append(evaluate_clustering(X_dense, true_labels, agg_st.fit_predict(X_dense),      'ST (PT-BR) + Agglomerative'))
results.append(evaluate_clustering(X_st_norm, true_labels, db_st.fit_predict(X_st_norm),   'ST (PT-BR) + DBSCAN'))
results.append(evaluate_clustering(X_dense, true_labels, hdb_st.fit_predict(X_dense),      'ST (PT-BR) + HDBSCAN'))

# Exibe tabela de resultados, ordenando por ARI decrescente
df_metrics = pd.DataFrame(results).sort_values('ARI', ascending=False).reset_index(drop=True)
df_metrics.style.background_gradient(subset=['ARI','NMI','Pureza'], cmap='RdYlGn', vmin=0, vmax=1)

In [ ]:
# Gráfico de barras comparativo — ARI por configuração
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

metrics_to_plot = ['ARI', 'Pureza']
colors = ['#7aa2f7', '#9ece6a']

for ax, metric, color in zip(axes, metrics_to_plot, colors):
    df_sorted = df_metrics.sort_values(metric, ascending=True)
    bars = ax.barh(df_sorted['Config'], df_sorted[metric],
                   color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    # Destaca o melhor
    max_idx = df_sorted[metric].idxmax()
    bars[list(df_sorted.index).index(max_idx)].set_color('#f7768e')
    ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=9)
    ax.set_xlabel(metric, fontsize=11)
    ax.set_xlim(0, 1.12)
    ax.set_title(f'Comparativo de {metric}', fontsize=12, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Métricas de Avaliação — TF-IDF vs Sentence Transformers PT-BR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Visualização 2D — PCA, t-SNE e UMAP

Reduzimos as representações de alta dimensão para 2D usando três técnicas:

| Técnica | Tipo | Preserva |
|:---|:---:|:---|
| **PCA** | Linear | Variância global |
| **t-SNE** | Não-linear | Estrutura local (vizinhança) |
| **UMAP** | Não-linear | Topologia local + global |

Plotamos tanto os **clusters preditos** quanto as **classes reais** para avaliar visualmente o alinhamento.

In [ ]:
# Obtemos os labels da melhor configuração (TF-IDF + Agglomerative)
best_labels = AgglomerativeClustering(n_clusters=6, linkage='ward').fit_predict(X_tfidf)

# Reduções 2D
print('⏳ PCA...')
coords_pca = PCA(n_components=2, random_state=42).fit_transform(X_tfidf)

print('⏳ t-SNE...')
perp = min(30, max(5, len(X_tfidf) // 10))
coords_tsne = TSNE(n_components=2, perplexity=perp, random_state=42,
                   init='pca', learning_rate='auto').fit_transform(X_tfidf)

print('⏳ UMAP...')
coords_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_tfidf)
print('✅ Redução concluída.')

In [ ]:
def scatter_panel(ax, coords, labels, title, is_category=False):
    """Plota um scatter 2D com legenda para clusters ou categorias reais."""
    cats = df['Categoria'].values
    unique = sorted(set(labels))
    for lbl in unique:
        mask = np.array(labels) == lbl
        if is_category:
            cat_name = cats[mask][0] if mask.any() else str(lbl)
            color = CATEGORY_COLORS.get(cat_name, '#888')
            label_str = cat_name
        else:
            color = CLUSTER_CMAP(lbl % 10)
            label_str = f'Cluster {lbl}' if lbl != -1 else 'Ruído'
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[color],
                   s=30, alpha=0.75, edgecolors='none', label=label_str)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, markerscale=1.2, loc='best', framealpha=0.7)
    ax.grid(linestyle='--', alpha=0.25)
    ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(3, 2, figsize=(13, 16), dpi=120)

for row_idx, (coords, proj_name) in enumerate([
    (coords_pca,  'PCA'),
    (coords_tsne, 't-SNE'),
    (coords_umap, 'UMAP'),
]):
    scatter_panel(axes[row_idx, 0], coords, best_labels,
                  f'{proj_name} — Clusters Preditos (TF-IDF + Agglomerative)')
    scatter_panel(axes[row_idx, 1], coords, df['Classe'].values,
                  f'{proj_name} — Classes Reais (Ground Truth)', is_category=True)

plt.suptitle('Projeção 2D — Clusters Preditos vs. Classes Reais (TF-IDF + Agglomerative)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Seleção de Landmarks — Medoides e Fronteiras

Para cada cluster, selecionamos duas categorias de amostras representativas:

1. **Medoides (Centrais):** documentos com **menor distância euclidiana ao centroide** do cluster.
   Representam o núcleo puro do tema — os textos mais "típicos" do grupo.

2. **Fronteiras (Ambíguos):** documentos com **menor coeficiente de silhueta individual** ($s \approx 0$ ou negativo).
   Estão na zona de decisão entre clusters — indicam interseções temáticas (ex.: notícia de economia com viés político).

Esses landmarks são exatamente o que alimenta o prompt enviado ao LLM para geração de rótulos semânticos.

In [ ]:
# Coeficiente de silhueta individual
sil_samples = silhouette_samples(X_tfidf, best_labels)

def get_landmarks(X, labels, sil_samples, cluster_id, n=3):
    """Retorna (medoides, fronteiras) para um cluster específico."""
    idxs = np.where(labels == cluster_id)[0]
    centroid = X[idxs].mean(axis=0)
    dists = np.linalg.norm(X[idxs] - centroid, axis=1)

    # Medoides: menor distância ao centroide
    medoid_local_idxs = np.argsort(dists)[:n]
    medoid_idxs = idxs[medoid_local_idxs]

    # Fronteiras: menor silhueta
    sil_local = sil_samples[idxs]
    frontier_local_idxs = np.argsort(sil_local)[:n]
    frontier_idxs = idxs[frontier_local_idxs]

    return medoid_idxs, frontier_idxs


# Análise do cluster alvo
cluster_target = 0
medoids, frontiers = get_landmarks(X_tfidf, best_labels, sil_samples, cluster_target)

print(f'=== ANÁLISE DE LANDMARKS — CLUSTER #{cluster_target} ===')
print(f'Categoria dominante: {df.iloc[np.where(best_labels == cluster_target)[0]]["Categoria"].value_counts().index[0]}\n')

print('--- MEDOIDES (NÚCLEO DO TEMA) ---')
for idx in medoids:
    row = df.iloc[idx]
    print(f' - ID: {idx:3d} | Silhueta: {sil_samples[idx]:+.3f} | Cat: {row["Categoria"]}')
    print(f'   Título: {row["Texto Original"][:100]}\n')

print('--- FRONTEIRAS (AMBÍGUOS / HÍBRIDOS) ---')
for idx in frontiers:
    row = df.iloc[idx]
    print(f' - ID: {idx:3d} | Silhueta: {sil_samples[idx]:+.3f} | Cat: {row["Categoria"]}')
    print(f'   Título: {row["Texto Original"][:100]}\n')

In [ ]:
# Visualização do perfil de silhueta por cluster (melhor config)
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
y_lower = 10
n_clusters = len(set(best_labels) - {-1})

for k in range(n_clusters):
    cluster_sil = np.sort(sil_samples[best_labels == k])
    size_k = cluster_sil.shape[0]
    y_upper = y_lower + size_k

    color = CLUSTER_CMAP(k / n_clusters)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + size_k / 2, f'C{k}', fontsize=9, ha='center', color='black')
    y_lower = y_upper + 10

avg_sil = sil_samples[best_labels != -1].mean()
ax.axvline(x=avg_sil, color='red', linestyle='--', linewidth=1.5,
           label=f'Média = {avg_sil:.3f}')
ax.set_xlabel('Coeficiente de Silhueta', fontsize=11)
ax.set_ylabel('Amostras por Cluster', fontsize=11)
ax.set_title('Perfil de Silhueta — TF-IDF + Agglomerative (Ward)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()